# 02: ChromaDB構築

前処理済みデータからChromaDBを構築します。

## 処理フロー

1. データ読み込み
2. テキスト分割
3. 埋め込み生成
4. ChromaDB保存

## 1. ライブラリインポート

In [1]:
import os
import json
import shutil
import torch

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

print("インポート完了")

インポート完了


## 2. 設定

In [2]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # 埋め込みモデル
USE_GPU = True  # GPU使用フラグ
FORCE_REBUILD = False  # True:強制再構築、False:既存DBがあればスキップ

device = "cuda" if (USE_GPU and torch.cuda.is_available()) else "cpu"  # デバイス判定
print(f"デバイス: {device}")
print(f"強制再構築: {FORCE_REBUILD}")

デバイス: cuda
強制再構築: False


## 3. データ読み込み

In [3]:
# データ読み込み
data_file = "../data/medquad_processed.json"  # 前処理済みデータ

with open(data_file, 'r', encoding='utf-8') as f:
    data = json.load(f)  # JSON読み込み

print(f"読み込み完了: {len(data)} サンプル")

読み込み完了: 16407 サンプル


## 4. ドキュメント作成

In [4]:
# Document形式に変換
documents = []

for item in data:
    doc = Document(
        page_content=item['content'],  # 本文
        metadata={
            "focus": item['focus'],  # トピック
            "source": item['source'],  # データソース
            "question": item['question']  # 元の質問
        }
    )
    documents.append(doc)

print(f"ドキュメント数: {len(documents)}")

ドキュメント数: 16407


## 5. テキスト分割

In [5]:
# テキスト分割
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # チャンクサイズ
    chunk_overlap=200,  # オーバーラップ（重複）
    separators=["\n\n", "\n", ". ", ", ", " ", ""]  # 区切り文字優先順位
)

chunks = text_splitter.split_documents(documents)  # 分割実行

print(f"チャンク数: {len(chunks)}")

チャンク数: 41179


## 6. 埋め込みモデル

In [6]:
# 埋め込みモデル初期化（RTX 3060最適化）
embeddings = HuggingFaceEmbeddings(
    model_name=MODEL_NAME,  # モデル名
    model_kwargs={'device': device},  # GPU/CPU指定
    encode_kwargs={
        'batch_size': 32,  # バッチ処理（GPU並列処理で高速化）
        'normalize_embeddings': True  # 正規化（精度向上）
    }
)

print(f"埋め込みモデル: {MODEL_NAME}")
print(f"バッチサイズ: 32（GPU最適化）")

埋め込みモデル: sentence-transformers/all-MiniLM-L6-v2
バッチサイズ: 32（GPU最適化）


## 7. ChromaDB構築

In [7]:
# ChromaDB構築（フラグ制御）
persist_directory = "../results/chroma_db"  # DB保存先

# 既存DBがある場合は確認
if os.path.exists(persist_directory) and not FORCE_REBUILD:
    print(f"既存DBが存在します: {persist_directory}")
    print("既存DBを読み込みます...")
    vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embeddings)
    print("読み込み完了")
else:
    # 古いDBを削除
    if os.path.exists(persist_directory):
        shutil.rmtree(persist_directory, ignore_errors=True)
        print(f"古いDBを削除")
    
    # ChromaDB構築
    print("ChromaDB構築中...")
    vectorstore = Chroma.from_documents(
        documents=chunks,  # ドキュメント
        embedding=embeddings,  # 埋め込みモデル
        persist_directory=persist_directory  # 保存先
    )
    print(f"構築完了: {persist_directory}")

既存DBが存在します: ../results/chroma_db
既存DBを読み込みます...


C:\Users\hello\AppData\Local\Temp\ipykernel_11884\2170415283.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embeddings)


読み込み完了


## 8. 検索テスト

In [8]:
# 既存DBを読み込み
if 'vectorstore' not in locals():
    print("既存DBを読み込みます...")
    vectorstore = Chroma(persist_directory="../results/chroma_db", embedding_function=embeddings)
    print("読み込み完了")
# 検索テスト

## 9. モジュール動作確認

In [9]:
# ChromaDB構築完了
print("ChromaDB構築完了")

ChromaDB構築完了


---

**次**: `03_demo.ipynb` でLLMを使った質問応答